# Stage 2. Participant grid

Audience: Ops & Eng, daily. Answers *who needs a phone call today*.

A heatmap ordered by risk score, the three-way slot breakdown that separates
participant disengagement from scheduler failure, the risk score broken into its
weighted terms, and the per-participant rail.

The invariant the grid turns on: a **structural blank** (a day outside the
participant's window) is not a **zero**. Blanks fall through to the page plane;
zeros are the palest step of the ramp.

In [1]:
# Every notebook under backend/dashboard/stages/ shares one data layer: the Django
# bootstrap, the SYNTHETIC_DATA switch, the fourteen frames and every compute
# function. Set SYNTHETIC_DATA in monitor_common.py to swap fixture for ORM.
from monitor_common import *

print("data source:", describe_source())

data source: Django ORM - c5dqhursbgn9fb.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com / dd28mps21fsprq


In [2]:
# STAGE 2A — CROSS-PARTICIPANT HEATMAP
# Risk at or above this gets a call today. A starting point, not a protocol
# number - it needs PI sign-off like every other threshold in this study.
RISK_ACTION_THRESHOLD = 10

METRIC_LABEL = {
    "slot_coverage": "Slot coverage (0-1)",
    "wear": "Wear coverage (0-1)",
    "delivered_n": "Prompts delivered (count/day)",
    "completeness_mean": "Item completeness (0-1)",
}
# delivered_n is the odd one out: dark means AT CAP, which is an alarm, while
# dark means good on the other three. Same ramp, opposite valence. It gets a
# diverging scale centred on the expected 2-3 per day instead.
METRIC_SCALE = {"delivered_n": "dose"}


def participant_order(phase="all"):
    """One canonical row order for every Stage 2 panel: risk descending.

    Three panels about the same people in three different sort orders makes the
    reader re-find each participant three times.
    """
    scores = risk_scores(phase)
    return [int(user_id) for user_id in scores["user_id"]], scores.set_index("user_id")


def participant_label(user_id, scores):
    """P1007 - 14  : identity, rank driver and score in the row label itself."""
    if user_id in scores.index and pd.notna(scores.at[user_id, "risk_score"]):
        return f"P{int(user_id)} · {int(scores.at[user_id, 'risk_score'])}"
    return f"P{int(user_id)} · -"


def alert_days_by_user():
    out = {}
    if alerts_open_df.empty:
        return out
    for uid, payload, fired_at in zip(
        alerts_open_df["user_id"], alerts_open_df["payload"], alerts_open_df["fired_at"]
    ):
        if pd.isna(uid):
            continue
        stamp = (payload or {}).get("date") or (payload or {}).get("local_date")
        local = pd.Timestamp(stamp).date() if stamp else local_date_of(fired_at)
        out.setdefault(int(uid), set()).add(local)
    return out


def grid(metric="slot_coverage", phase="all", axis="study_day"):
    if metric not in GRID_METRICS:
        raise ValueError(f"metric must be one of {GRID_METRICS}")
    days = daily_grid_metrics(phase)
    order, scores = participant_order(phase)
    flagged = alert_days_by_user()

    if axis == "study_day":
        columns = list(range(STUDY_DAYS))
        key_of = lambda row: row["study_day"]
    else:
        columns = sorted(set(days["local_date"]))
        key_of = lambda row: row["local_date"]

    rows = []
    for uid in order:
        own = days[days["user_id"].eq(uid)]
        by_key = {key_of(row): row for row in own.to_dict("records")}
        values, run_in, silent, alert, dates = [], [], [], [], []
        for column in columns:
            row = by_key.get(column)
            values.append(None if row is None else (None if pd.isna(row[metric]) else float(row[metric])))
            run_in.append(None if row is None else bool(row["is_run_in"]))
            silent.append(None if row is None else int(row["slots_silent"]) > 0)
            dates.append(None if row is None else row["local_date"])
            alert.append(False if row is None else row["local_date"] in flagged.get(uid, set()))
        rows.append({
            "user_id": uid,
            "label": participant_label(uid, scores),
            "risk_score": scores.at[uid, "risk_score"] if uid in scores.index else None,
            "risk_components": scores.at[uid, "risk_components"] if uid in scores.index else None,
            "phase": scores.at[uid, "phase"] if uid in scores.index else None,
            "values": values, "run_in": run_in, "silent": silent, "alert": alert,
            "local_dates": dates,
        })
    return {"metric": metric, "axis": axis, "phase": phase, "columns": columns,
            "benchmark": {"slot_coverage": BENCHMARKS["slot_coverage"],
                          "wear": BENCHMARKS["wear"],
                          "delivered_n": DAILY_PROMPT_CAP,
                          "completeness_mean": None}[metric],
            "rows": rows}


def grid_frame(metric="slot_coverage", phase="all", axis="study_day"):
    payload = grid(metric, phase, axis)
    return pd.DataFrame([row["values"] for row in payload["rows"]],
                        index=[row["label"] for row in payload["rows"]],
                        columns=payload["columns"])


def _metric_trace(payload, metric, visible, axis, show_values):
    zmax = DAILY_PROMPT_CAP if metric == "delivered_n" else 1.0
    scale_key = METRIC_SCALE.get(metric)
    if scale_key:
        colorscale, zmin = ink(scale_key), 0
    else:
        # Four discrete steps rather than a continuous ramp: slot coverage takes
        # seven values and a smooth ramp wastes the encoding. Seven steps cannot
        # clear the adjacent-lightness floor, so the exact count rides in the
        # cell text instead.
        steps = ink("heatmap_steps")
        colorscale, zmin = [], 0
        for index, colour in enumerate(steps):
            colorscale.append([index / len(steps), colour])
            colorscale.append([(index + 1) / len(steps), colour])
    text = None
    if show_values and metric in ("delivered_n", "slot_coverage"):
        factor = SCHEDULED_CHECK_IN_DAILY_CAP if metric == "slot_coverage" else 1
        # Empty string, not None: Plotly renders a None cell as the literal
        # text "null", which would print across every structural blank.
        text = [["" if value is None else f"{round(value * factor):g}"
                 for value in row["values"]] for row in payload["rows"]]
    return go.Heatmap(
        z=[row["values"] for row in payload["rows"]],
        x=[str(column) for column in payload["columns"]],
        y=[row["label"] for row in payload["rows"]],
        zmin=zmin, zmax=zmax, colorscale=colorscale, visible=visible,
        xgap=2, ygap=2, hoverongaps=False,
        text=text, texttemplate="%{text}" if text else None,
        textfont=dict(size=8, color=ink("text") if scale_key else ink("surface")),
        colorbar=dict(title=dict(text=METRIC_LABEL[metric], side="right", font=dict(size=11)),
                      thickness=10, outlinewidth=0, tickfont=dict(size=10)),
        customdata=[[str(date) for date in row["local_dates"]] for row in payload["rows"]],
        hovertemplate=("%{y}<br>" + ("study day" if axis == "study_day" else "date")
                       + " %{x}<br>%{customdata}<br>" + METRIC_LABEL[metric]
                       + ": %{z}<extra></extra>"))


def plot_grid(metric="slot_coverage", phase="all", axis="study_day", show_values=True):
    payloads = {name: grid(name, phase, axis) for name in GRID_METRICS}
    base = payloads[metric]
    rows = base["rows"]
    labels = [row["label"] for row in rows]
    columns = base["columns"]
    x = [str(column) for column in columns]

    fig = make_subplots(rows=2, cols=2, shared_xaxes=True, shared_yaxes=True,
                        row_heights=[0.12, 0.88], column_widths=[0.93, 0.07],
                        vertical_spacing=0.02, horizontal_spacing=0.012)

    for name in GRID_METRICS:
        fig.add_trace(_metric_trace(payloads[name], name, name == metric, axis, show_values),
                      row=2, col=1)

    # Marginal summaries. The column strip is how a cohort-wide outage shows up
    # without remembering to flip to the calendar axis: a Celery failure hits
    # everyone on one date, so it appears here as a single dark or empty column.
    column_mean = []
    for position in range(len(columns)):
        present = [row["values"][position] for row in rows if row["values"][position] is not None]
        column_mean.append(sum(present) / len(present) if present else None)
    fig.add_trace(go.Heatmap(
        z=[column_mean], x=x, y=["cohort mean"], zmin=0,
        zmax=DAILY_PROMPT_CAP if metric == "delivered_n" else 1.0,
        colorscale=ink(METRIC_SCALE.get(metric, "heatmap")), showscale=False,
        xgap=2, ygap=2, hoverongaps=False,
        hovertemplate="cohort mean<br>%{x}: %{z:.2f}<extra></extra>"), row=1, col=1)

    row_mean = []
    for row in rows:
        present = [value for value in row["values"] if value is not None]
        row_mean.append(sum(present) / len(present) if present else None)
    fig.add_trace(go.Heatmap(
        z=[[value] for value in row_mean], x=["mean"], y=labels, zmin=0,
        zmax=DAILY_PROMPT_CAP if metric == "delivered_n" else 1.0,
        colorscale=ink(METRIC_SCALE.get(metric, "heatmap")), showscale=False,
        xgap=2, ygap=2, hoverongaps=False,
        hovertemplate="%{y}<br>participant mean: %{z:.2f}<extra></extra>"), row=2, col=2)

    if axis == "study_day":
        run_in_end = min(RUN_IN_DAYS, len(columns)) - 0.5
        fig.add_shape(type="rect", x0=-0.5, x1=run_in_end, y0=-0.5, y1=len(rows) - 0.5,
                      line=dict(width=0), fillcolor=ink("band"), opacity=0.55, layer="below",
                      row=2, col=1)

    # Silent slots stay on the cells; open alerts move to their own gutter
    # column, so two marker shapes in the same status colour no longer compete
    # on top of the ramp.
    silent_x, silent_y = [], []
    alert_y, alert_text = [], []
    for row in rows:
        hits = 0
        for position in range(len(columns)):
            if row["silent"][position]:
                silent_x.append(x[position]); silent_y.append(row["label"])
            hits += int(bool(row["alert"][position]))
        if hits:
            alert_y.append(row["label"]); alert_text.append(str(hits))
    if silent_x:
        fig.add_trace(go.Scatter(
            x=silent_x, y=silent_y, mode="markers", name="silent slot",
            marker=dict(symbol="line-ne", size=11, color=STATUS["critical"],
                        line=dict(width=2, color=STATUS["critical"])),
            hovertemplate="scheduler silent<extra></extra>", showlegend=True),
            row=2, col=1)
    if alert_y:
        fig.add_trace(go.Scatter(
            x=["alerts"] * len(alert_y), y=alert_y, mode="markers+text",
            name="open alert days", text=alert_text, textposition="middle center",
            textfont=dict(size=9, color=ink("surface")),
            marker=dict(symbol="square", size=16, color=STATUS["critical"]),
            hovertemplate="%{y}<br>%{text} day(s) with an open alert<extra></extra>",
            showlegend=True), row=2, col=2)

    # Everything above this rule is today's call list.
    above = [index for index, row in enumerate(rows)
             if row["risk_score"] is not None and row["risk_score"] >= RISK_ACTION_THRESHOLD]
    if above:
        boundary = max(above) + 0.5
        for column in (1, 2):
            fig.add_shape(type="line", x0=0, x1=1, xref="x domain" if column == 1 else "x2 domain",
                          y0=boundary, y1=boundary,
                          line=dict(color=STATUS["critical"], width=2, dash="dash"),
                          row=2, col=column)
        fig.add_annotation(x=1, xref="x domain", xanchor="right", y=boundary, yshift=9,
                           text=f"call today (risk >= {RISK_ACTION_THRESHOLD})", showarrow=False,
                           font=dict(size=10, color=STATUS["critical"]), row=2, col=1)

    buttons = []
    for index, name in enumerate(GRID_METRICS):
        visible = [position == index for position in range(len(GRID_METRICS))]
        visible += [True] * (len(fig.data) - len(GRID_METRICS))
        buttons.append(dict(label=METRIC_LABEL[name], method="update",
                            args=[{"visible": visible}]))

    height = max(360, 34 * len(rows) + 230)
    base_layout(fig, height,
                f"Participant grid - {phase} ({'study day' if axis == 'study_day' else 'calendar date'})",
                margin=dict(l=130, r=110, t=104, b=54),
                context={"phase": phase, "n": len(rows)})
    fig.update_layout(
        plot_bgcolor=ink("page"), showlegend=bool(silent_x or alert_y),
        legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom",
                    font=dict(size=10, color=ink("text_secondary"))),
        updatemenus=[dict(buttons=buttons, direction="down", showactive=True,
                          active=list(GRID_METRICS).index(metric),
                          x=1.0, xanchor="right", y=1.13, yanchor="top",
                          bgcolor=ink("surface"), bordercolor=ink("axis"),
                          font=dict(size=11, color=ink("text")))])
    fig.update_xaxes(title=dict(text="study day" if axis == "study_day" else "local date",
                                font=dict(size=11, color=ink("muted"))),
                     type="category", tickangle=0, nticks=18, row=2, col=1)
    fig.update_xaxes(type="category", showticklabels=False, row=1, col=1)
    fig.update_xaxes(type="category", showticklabels=True, tickfont=dict(size=9), row=2, col=2)
    fig.update_yaxes(type="category", autorange="reversed",
                     title=dict(text="participant · risk score (desc)",
                                font=dict(size=11, color=ink("muted"))), row=2, col=1)
    fig.update_yaxes(type="category", showticklabels=False, row=1, col=1)
    return fig


def plot_grid_small_multiples(phase="all", axis="study_day"):
    """All four metrics as stacked rows - the dropdown is dead in a PNG or PDF."""
    payloads = {name: grid(name, phase, axis) for name in GRID_METRICS}
    rows = payloads[GRID_METRICS[0]]["rows"]
    labels = [row["label"] for row in rows]
    fig = make_subplots(rows=len(GRID_METRICS), cols=1, shared_xaxes=True,
                        vertical_spacing=0.035,
                        subplot_titles=[METRIC_LABEL[name] for name in GRID_METRICS])
    for index, name in enumerate(GRID_METRICS, start=1):
        trace = _metric_trace(payloads[name], name, True, axis, False)
        trace.update(showscale=False)
        fig.add_trace(trace, row=index, col=1)
        fig.update_yaxes(type="category", autorange="reversed",
                         tickfont=dict(size=9, color=ink("text")), row=index, col=1)
    fig.update_xaxes(type="category", nticks=18,
                     title=dict(text="study day" if axis == "study_day" else "local date",
                                font=dict(size=11, color=ink("muted"))),
                     row=len(GRID_METRICS), col=1)
    base_layout(fig, max(700, 22 * len(labels) * len(GRID_METRICS) + 180),
                f"Participant grid - all four metrics - {phase}",
                margin=dict(l=130, r=40, t=86, b=54),
                context={"phase": phase, "n": len(rows)})
    for annotation in fig.layout.annotations[:len(GRID_METRICS)]:
        annotation.update(x=0, xanchor="left", font=dict(size=11, color=ink("text")))
    return fig

In [3]:
# STAGE 2B/2C/2D — SLOT SPLIT, RISK TABLE, RAIL
# All three share participant_order(), so a participant sits at the same height
# in every Stage 2 panel.
SLOT_CATEGORIES = (
    ("covered", "Covered - check-in submitted", "series_1"),
    ("reminded", "Reminded, not covered - participant skipped", "series_3"),
    ("silent", "Silent - scheduler never fired (not non-compliance)", "series_2"),
)
RISK_TERM_LABEL = {
    "ema_stale": "days since last check-in",
    "low_coverage": "low slot coverage",
    "sync_stale": "stale sync",
    "low_wear": "low wear",
    "missing_signal": "missing B1/B2",
    "open_critical": "open alert",
}


def _ordered_labels(phase):
    order, scores = participant_order(phase)
    return order, scores, [participant_label(uid, scores) for uid in order]


def _action_rule(fig, rows, scores, order, col=1, xref="x domain"):
    above = [index for index, uid in enumerate(order)
             if pd.notna(scores.at[uid, "risk_score"])
             and scores.at[uid, "risk_score"] >= RISK_ACTION_THRESHOLD]
    if not above:
        return
    boundary = max(above) + 0.5
    fig.add_shape(type="line", x0=0, x1=1, xref=xref, y0=boundary, y1=boundary,
                  line=dict(color=STATUS["critical"], width=2, dash="dash"))
    fig.add_annotation(x=1, xref=xref, xanchor="right", y=boundary, yshift=9,
                       text=f"call today (risk >= {RISK_ACTION_THRESHOLD})", showarrow=False,
                       font=dict(size=10, color=STATUS["critical"]))


def plot_slot_split(phase="all", days=RISK_TRAILING_DAYS):
    split = slot_split(phase, days)
    order, scores, labels = _ordered_labels(phase)
    fig = go.Figure()
    if split.empty:
        fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           text="no active participant-days in the window",
                           font=dict(size=12, color=ink("muted")))
        base_layout(fig, 200, f"Slot breakdown - trailing {days} days",
                    context={"phase": phase, "n": 0})
        return fig

    split = split.set_index("user_id").reindex([uid for uid in order if uid in set(split["user_id"])])
    split = split.dropna(subset=["covered_pct"])
    rows = [participant_label(uid, scores) for uid in split.index]
    for column, legend, role in SLOT_CATEGORIES:
        fig.add_trace(go.Bar(
            x=split[f"{column}_pct"], y=rows, orientation="h", name=legend,
            marker=dict(color=ink(role), line=dict(color=ink("surface"), width=2)),
            text=[f"{value:.0%}" if value >= 0.08 else "" for value in split[f"{column}_pct"]],
            textposition="inside", insidetextanchor="middle",
            textfont=dict(size=10, color=ink("surface")),
            customdata=split[column],
            hovertemplate=(f"%{{y}}<br>{legend}<br>"
                           "%{customdata} slots (%{x:.0%})<extra></extra>")))

    _action_rule(fig, rows, scores, [uid for uid in split.index])
    base_layout(fig, max(280, 30 * len(split) + 170),
                f"Check-in slots by outcome - trailing {days} days",
                margin=dict(l=130, r=40, t=126, b=54),
                context={"phase": phase, "n": len(split)})
    fig.update_layout(barmode="stack", bargap=0.3, showlegend=True,
                      legend=dict(orientation="h", y=1.04, x=0, yanchor="bottom",
                                  font=dict(size=10, color=ink("text_secondary"))))
    fig.update_xaxes(range=[0, 1], tickformat=".0%",
                     title=dict(text="share of the day's six slots",
                                font=dict(size=11, color=ink("muted"))))
    fig.update_yaxes(type="category", categoryorder="array", categoryarray=rows[::-1],
                     title=dict(text="participant · risk score (desc)",
                                font=dict(size=11, color=ink("muted"))))
    return fig


def plot_risk_contributions(phase="all"):
    """Participant x term heat table with the total as a bar on the right.

    A six-category stacked bar was the weakest chart in the set: six hues
    carrying meaning, three of them below 3:1 on the light surface. A heat table
    reads faster, matches the grid idiom next to it, and needs no hue at all.
    """
    order, scores, _labels = _ordered_labels(phase)
    scored = scores[scores["risk_score"].notna()]
    order = [uid for uid in order if uid in scored.index]
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.62, 0.38],
                        horizontal_spacing=0.02,
                        subplot_titles=("points by term", "total risk score"))
    if not order:
        fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           text="no participant is currently in run-in or MRT",
                           font=dict(size=12, color=ink("muted")))
        base_layout(fig, 200, "Risk score contributions", context={"phase": phase, "n": 0})
        return fig

    terms = list(RISK_WEIGHTS)
    rows = [participant_label(uid, scores) for uid in order]
    z = [[scored.at[uid, "risk_components"].get(term, 0) for term in terms] for uid in order]
    fig.add_trace(go.Heatmap(
        z=z, x=[RISK_TERM_LABEL[term] for term in terms], y=rows,
        zmin=0, zmax=max(1, max(max(row) for row in z)),
        colorscale=[[0.0, ink("page")], [0.001, ink("heatmap_steps")[0]],
                    [1.0, ink("heatmap_steps")[-1]]],
        xgap=2, ygap=2, showscale=False,
        text=[[str(value) if value else "" for value in row] for row in z],
        texttemplate="%{text}", textfont=dict(size=10, color=ink("surface")),
        hovertemplate="%{y}<br>%{x}: %{z} points<extra></extra>"), row=1, col=1)
    totals = [int(scored.at[uid, "risk_score"]) for uid in order]
    fig.add_trace(go.Bar(
        x=totals, y=rows, orientation="h",
        marker=dict(color=ink("series_1"), line=dict(color=ink("surface"), width=2)),
        text=[str(total) for total in totals], textposition="outside",
        textfont=dict(size=10, color=ink("text")),
        hovertemplate="%{y}<br>risk %{x} of 47<extra></extra>",
        showlegend=False), row=1, col=2)

    _action_rule(fig, rows, scores, order, xref="x domain")
    base_layout(fig, max(300, 32 * len(order) + 180), f"Risk score contributions - {phase}",
                margin=dict(l=130, r=50, t=96, b=64),
                context={"phase": phase, "n": len(order)})
    fig.update_xaxes(tickangle=-25, tickfont=dict(size=9, color=ink("muted")),
                     title=dict(text="weighted term", font=dict(size=11, color=ink("muted"))),
                     row=1, col=1)
    fig.update_xaxes(rangemode="tozero",
                     title=dict(text="points (max 47)", font=dict(size=11, color=ink("muted"))),
                     row=1, col=2)
    fig.update_yaxes(type="category", categoryorder="array", categoryarray=rows[::-1],
                     title=dict(text="participant · risk score (desc)",
                                font=dict(size=11, color=ink("muted"))), row=1, col=1)
    for annotation in fig.layout.annotations[:2]:
        annotation.update(font=dict(size=11, color=ink("text")))
    return fig


def style_participant_rail(user_id, phase="all"):
    rail = participant_rail(user_id, phase)
    sync = ("no writer - unmeasurable" if not rail["sync_measurable"]
            else "-" if rail["last_sync_age_h"] is None
            else f"{rail['last_sync_age_h']:.1f} h ago")
    coverage = (f"{rail['slot_coverage_num']}/{rail['slot_coverage_den']}"
                + (f" ({rail['slot_coverage_rate']:.0%})" if rail["slot_coverage_rate"] is not None
                   else "  rate withheld"))
    rows = [
        ("participant", str(rail["user_id"])),
        ("phase", rail["phase"] or "-"),
        ("study day", f"{rail['study_day']} of {STUDY_DAYS - 1}"),
        ("days remaining", str(rail["days_remaining"])),
        ("risk score", "-" if rail["risk_score"] is None else
         f"{int(rail['risk_score'])} of 47"),
        ("call today?", "yes" if (rail["risk_score"] or 0) >= RISK_ACTION_THRESHOLD else "no"),
        ("last check-in", "-" if rail["last_ema_at"] is None
         else pd.Timestamp(rail["last_ema_at"]).strftime("%b %d %H:%M")),
        ("last sync", sync),
        ("slot coverage", coverage),
        ("wear days met", f"{rail['wear_days_met']}/{rail['wear_days_scored']}"),
        ("prompts delivered", str(rail["prompts_delivered"])),
        ("open alerts", str(len(rail["open_alerts"]))),
    ]
    table = pd.DataFrame(rows, columns=["field", "value"])
    return (table.style
            .set_properties(**{"font-family": FONT, "font-size": "12px",
                               "color": ink("text"), "background-color": ink("surface")})
            .set_properties(subset=["field"], **{"color": ink("muted"), "font-size": "11px"})
            .set_table_styles([
                {"selector": "th", "props": [("display", "none")]},
                {"selector": "td", "props": [("border-bottom", f"1px solid {ink('grid')}"),
                                             ("padding", "4px 10px")]}])
            .hide(axis="index"))

## 2A - Cross-participant heatmap

In [4]:
# Rows are participants ordered by risk DESCENDING - the same order every other
# Stage 2 panel uses, so a participant sits at the same height throughout. Row
# labels carry the score (P1003 · 31), so the grid explains its own ordering.
# The dashed rule is today's call list: everything above it is risk >= 10.
# New marginals: the strip above is the cohort mean per column, which is how a
# Celery outage shows up (one dark or empty column) without flipping to the
# calendar axis; the narrow column on the right is each participant's own mean.
# Open alerts moved into their own gutter column so two marker shapes in the
# same red no longer compete on top of the ramp; cells keep the silent-slot
# slash only. Slot coverage cells carry the slot count as a numeral, because a
# seven-level ramp cannot be read back to an exact count from hue alone.
plot_grid("slot_coverage", "all")

In [5]:
# Prompts delivered on its own DIVERGING scale, not the shared blue ramp. On
# the other three metrics dark means good; on this one dark means at the daily
# cap, which is an alarm condition. Blue is under-dosed, neutral is the expected
# 2-3 per day, red is at cap - so both ends read as problems and the middle
# reads as nothing.
plot_grid("delivered_n", "all")

In [6]:
# The same grid on a CALENDAR axis. With the cohort-mean strip above this is
# now optional rather than load-bearing, but it is still the clearest way to see
# a cohort-wide outage: everyone drops on one date, not on one study day.
plot_grid("wear", "all", axis="local_date")

In [7]:
# All four metrics stacked, for export. The dropdown above is dead in a PNG or
# a PDF, so the report view needs every metric on the page at once.
plot_grid_small_multiples("all")

## 2B - Slot breakdown

In [8]:
# Each participant's six daily slots over the trailing week, split three ways:
# covered, reminded-but-skipped, and silent. Silent is a SYSTEM failure - Celery
# down, dead push token, is_enrolled flipped - not non-compliance, and nobody
# should be phoned about it.
# Now in risk order rather than sorted by silent share, so this panel lines up
# with the grid above and the risk table below. P1003 is 100% silent (42 of 42
# slots): the pipeline never asked it for anything, which is also why it tops
# the risk list.
plot_slot_split("all")

## 2C - Risk score

In [9]:
# The risk score as a participant x term heat table, with the total as a bar on
# the right. This replaces a six-category stacked bar: six hues carrying meaning
# is the weakest encoding in the set, and three of those hues sat below 3:1 on
# the light surface. A heat table needs no hue at all, matches the grid idiom
# next to it, and the numerals make every term exact.
# P1003's 31 points are almost all stale check-ins (15) and low coverage (14).
plot_risk_contributions("all")

## 2D - Participant rail

In [10]:
# The right rail for one participant: phase, study day, days remaining, sync
# freshness, cumulative benchmark rates with their raw counts, and open alerts.
# last sync reads "no writer - unmeasurable" whenever nothing advances the sync
# clock, rather than showing a misleading zero age.
GRID_USER = int(risk_scores("all")["user_id"].iloc[0])
style_participant_rail(GRID_USER)

field,value
participant,430
phase,mrt
study day,28 of 34
days remaining,6
risk score,33 of 47
call today?,yes
last check-in,Sep 11 04:38
last sync,no writer - unmeasurable
slot coverage,2/174 rate withheld
wear days met,0/0


In [12]:
# The numbers behind the grid: one row per participant-day with every metric the
# heatmap can colour by.
display(daily_grid_metrics("all"))

,user_id,local_date,study_day,is_run_in,slots_covered,slots_reminded_uncovered,slots_silent,slots_reminded_by_index,gap_coverage,minute_coverage,gap_minutes,delivered_n,decision_points_n,eligible_n,sent_n,completeness_mean,ema_missing_b1b2_n,slot_coverage,slots_expected,wear
0,430,2026-08-21,0,True,0,0,6,0,NaN,NaN,NaN,1,1,0,1,1.0,0,0.000000,6,NaN
1,430,2026-08-22,1,True,0,0,6,0,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
2,430,2026-08-23,2,True,0,0,6,0,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
3,430,2026-08-24,3,True,0,0,6,0,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
4,430,2026-08-25,4,True,0,0,6,0,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
5,430,2026-08-26,5,True,0,0,6,0,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
6,430,2026-08-27,6,True,2,4,0,2,NaN,NaN,NaN,1,6,0,3,1.0,0,0.333333,6,NaN
7,430,2026-08-28,7,False,0,6,0,1,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
8,430,2026-08-29,8,False,0,6,0,1,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
9,430,2026-08-30,9,False,0,6,0,1,NaN,NaN,NaN,0,0,0,0,NaN,0,0.000000,6,NaN
